# 02 — Offline Preprocessing

Tạo cache dùng chung cho **cả hai pipeline** (Faster R-CNN và YOLO26m).  
Owner: **Team analysis + preprocessing offline**.

Quy trình tiền xử lý chuẩn 5 BƯỚC:
1. **BƯỚC 1: Xóa các box nhỏ** (< 8px trên canvas 640×640)
2. **BƯỚC 2: Tính gom cụm và chia** (dHash 1024-bit chuẩn hóa)
3. **BƯỚC 3: Đảm bảo train luôn có đầy đủ class** (52/52 lớp, 100% Cluster Isolation)
4. **BƯỚC 4: Tạo augment trong train** (cân bằng nhãn thiểu số, max 7 ảnh/lớp, blur nhẹ, không xoay)
5. **BƯỚC 5: Resize ảnh** (Letterbox 640×640, pad 114) & xuất data.yaml, meta.json

> **Quan trọng**: Không sửa data/processed/ sau khi tạo. Hai team pipeline đọc chung cache này. Online augmentation là switch bật/tắt riêng trong lúc tune.


## 0) Bootstrap

Chạy trên local hoặc Kaggle/Colab. Trên Kaggle bật **Internet: On**.


In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL    = 'https://github.com/tuan8p/VN-Traffic-Sign-Detection.git'
REPO_DIR    = 'VN-Traffic-Sign-Detection'
REPO_BRANCH = 'preprocessing'  # nhánh của Team analysis

base = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
repo_path = base / REPO_DIR

def sh(cmd):
    print('>>', cmd)
    subprocess.run(cmd, shell=True, check=True)

if (Path.cwd() / '.git').exists() and (Path.cwd() / 'configs' / 'shared.yaml').exists():
    repo_path = Path.cwd()
    print('Using the repo this notebook sits in:', repo_path)
elif not (repo_path / '.git').exists():
    sh(f'git clone --branch {REPO_BRANCH} {REPO_URL} "{repo_path}"')

git = f'git -C "{repo_path}"'
sh(f'{git} fetch origin {REPO_BRANCH}')
sh(f'{git} checkout -B {REPO_BRANCH} origin/{REPO_BRANCH}')

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print('cwd:', Path.cwd())

sh('pip install -q -e "."')
print('Bootstrap OK.')

## 1) Cấu hình

Trên Kaggle: **Add Input** → dataset thô → điền path vào `DATA_ROOT_RAW`.


In [ ]:
from pathlib import Path
import os

# ── THAY ĐỔI BIẾN NÀY ─────────────────────────────────────────
DATA_ROOT_RAW = os.environ.get('DATA_ROOT') or 'data/raw'
# Ví dụ Kaggle: '/kaggle/input/vnts-raw'
# ──────────────────────────────────────────────────────────────

root_raw = Path(DATA_ROOT_RAW).resolve()
assert root_raw.exists(), f'[!] Không tìm thấy DATA_ROOT_RAW: {root_raw}'

print(f'Dataset thô : {root_raw}')

# Kiểm tra nhanh
img_dir = root_raw / 'images'
lbl_dir = root_raw / 'labels'
n_imgs = len(list(img_dir.glob('*.*'))) if img_dir.is_dir() else 0
n_lbls = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.is_dir() else 0
print(f'Ảnh gốc     : {n_imgs}')
print(f'File nhãn   : {n_lbls}')

if n_imgs == 0:
    print('[!] Chưa thấy ảnh. Kiểm tra đường dẫn DATA_ROOT_RAW.')

## 2) Resolve config


In [ ]:
from vn_tsd.config.resolve import resolve_config

cfg = resolve_config(shared_yaml='configs/shared.yaml')

pre = cfg.get('preprocess', {})
print('=== Cấu hình preprocess ===')
print(f'  image_size : {pre.get("image_size")}')
print(f'  keep_aspect: {pre.get("keep_aspect")}')
print(f'  pad_value  : {pre.get("pad_value")}')
print(f'  min_box_px : {pre.get("min_box_px")}')
print(f'  hflip      : {pre.get("hflip")}  ← tắt do biển đối xứng gương')

sp = cfg.get('split', {})
print(f'  split policy: use_provided={sp.get("use_provided_splits")}, ratios={sp.get("ratios")}')
print(f'  seed       : {cfg.get("seed")}')

## 3) Chạy preprocessing offline

Ô này sẽ:
- Letterbox toàn bộ ảnh về 640×640
- Phân chia train/val/test
- Lọc tiny box
- Ghi `data/processed/` + `data.yaml` + `meta.json`


In [ ]:
from vn_tsd.data.preprocess_offline import run_offline

out_root = run_offline(cfg, DATA_ROOT_RAW)
print(f'\n[OK] Processed cache: {out_root}')

## 4) Kiểm tra kết quả


In [ ]:
import json
from pathlib import Path

out_root = Path('data/processed')

# meta.json
meta_path = out_root / 'meta.json'
if meta_path.exists():
    with open(meta_path) as f:
        meta = json.load(f)
    print('=== meta.json ===')
    print(json.dumps(meta, indent=2, ensure_ascii=False))
else:
    print('[!] meta.json chưa có')

# data.yaml
yaml_path = out_root / 'data.yaml'
print(f'\ndata.yaml tồn tại: {yaml_path.exists()}')

# Đếm file
for split in ('train', 'val', 'test'):
    img_n = len(list((out_root / 'images' / split).glob('*.*'))) if (out_root / 'images' / split).is_dir() else 0
    lbl_n = len(list((out_root / 'labels' / split).glob('*.txt'))) if (out_root / 'labels' / split).is_dir() else 0
    print(f'  {split}: {img_n} ảnh, {lbl_n} file nhãn')

## 5) Xem mẫu ảnh đã letterbox


In [ ]:
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

out_root = Path('data/processed')
train_img_dir = out_root / 'images' / 'train'
train_lbl_dir = out_root / 'labels' / 'train'

img_files = sorted(train_img_dir.glob('*.jpg')) if train_img_dir.is_dir() else []
if not img_files:
    print('[!] Chưa có ảnh trong data/processed/images/train/')
else:
    samples = random.sample(img_files, min(6, len(img_files)))
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('Mẫu ảnh sau letterbox 640×640 (màu box = class_id)', fontsize=12, fontweight='bold')

    COLORS = plt.cm.tab20(np.linspace(0, 1, 52))

    for ax, img_path in zip(axes.flat, samples):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        lbl_path = train_lbl_dir / (img_path.stem + '.txt')
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    parts = line.split()
                    if len(parts) < 5:
                        continue
                    cid = int(parts[0])
                    cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                    x1 = int((cx - bw / 2) * w)
                    y1 = int((cy - bh / 2) * h)
                    x2 = int((cx + bw / 2) * w)
                    y2 = int((cy + bh / 2) * h)
                    color = tuple(int(c * 255) for c in COLORS[cid % 20][:3])
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(img, str(cid), (x1, max(y1 - 3, 0)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

        ax.imshow(img)
        ax.axis('off')
        ax.set_title(img_path.stem[:20], fontsize=8)

    plt.tight_layout()
    plt.show()
    print(f'Hiển thị {len(samples)} mẫu ảnh với bounding box.')

## 6) Chạy EDA lần 2 (trên data đã processed)

Sau khi preprocessing xong, chạy lại EDA để kiểm tra cache có đồng đều không.


In [ ]:
from analysis.eda import main as eda_main

eda_main([
    '--data-root',   'data/processed',
    '--figures-dir', 'analysis/figures',
    '--class-table', 'configs/classes.csv',
    '--image-size',  '640',
    '--min-box-px',  '8',
])

print('\n[OK] EDA lần 2 trên data/processed hoàn thành. Xem hình tại analysis/figures/')

## 7) Xem hình EDA sau preprocessing


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig_dir = Path('analysis/figures')
# Lọc hình của lần 2 (tag = processed)
figs = sorted(fig_dir.glob('*processed*.png'))
if not figs:
    figs = sorted(fig_dir.glob('*.png'))  # fallback: tất cả

for fpath in figs:
    img = mpimg.imread(fpath)
    plt.figure(figsize=(16, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(fpath.stem, fontsize=11)
    plt.tight_layout()
    plt.show()

## 8) Upload lên Kaggle Dataset (để team YOLO/FasterRCNN dùng)

Sau khi chạy thành công, nén `data/processed/` và tải lên Kaggle Dataset để các team pipeline add vào notebook của họ.

```bash
# Tạo dataset metadata
kaggle datasets init -p data/processed
# Chỉnh sửa data/processed/dataset-metadata.json: title, id, ...
# Upload
kaggle datasets create -p data/processed
# Hoặc update version mới:
kaggle datasets version -p data/processed -m 'letterbox 640x640, min_box_px=8'
```

Sau đó thông báo các team pipeline path dataset để điền vào `CANDIDATES` trong notebook 03 và 04.
